# Colab の T4 で privacy-masker の処理速度を測る

PyPI の最新版を入れ、架空の日本語文書 500 件で CPU と GPU の処理速度を比べる。文書はこのノートブック内で合成する（Google Drive や社内データは不要）。

Colab で開き、**ランタイム → ランタイムのタイプを変更 → T4 GPU** を選んで、上から順に実行する。GPU が無いときは CPU だけ測る。

`CharHeadNerDetector` の `device` の既定は `"cpu"`。**GPU は自動では使われない**ので、`device="cuda"` を明示する。detector 側の `batch_size` は CPU で 8〜16、GPU で 32 以上が目安（README の性能の目安と同じ）。ここでは detector と `transform_batch()` の両方に同じ値を渡す。

In [ ]:
# Colab に入っている torch で確認する。nvidia-smi が無くても次へ進む。
!nvidia-smi
import torch

HAS_CUDA = torch.cuda.is_available()
print(f"torch: {torch.__version__}")
print(f"torch.cuda.is_available(): {HAS_CUDA}")
print(f"デバイス: {torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU（GPU の計測は飛ばす）'}")

In [ ]:
# バージョンを固定せず、PyPI の最新版を使う。
%pip install -q 'privacy-masker[transformers]' --upgrade
import privacy_masker

print(f"privacy-masker: {privacy_masker.__version__}")

In [ ]:
import os
import random

# ローカルの動作確認時だけ環境変数で減らせる。通常は 500 文書。
N_DOCS = int(os.environ.get("PRIVACY_MASKER_BENCH_DOCS", "500"))
if N_DOCS < 1:
    raise ValueError("文書数は 1 以上にする")
rng = random.Random(20261103)

# 人名・社名・地名は架空。電話は 000 始まりのダミー、メールは例示用ドメイン。
people = ["雲綴 ポルナ", "星架 ネムリオ", "夢織 プラノ", "虹紡 モフリナ"]
companies = ["株式会社架空ポルネ工房", "株式会社架空ネムポ配送", "架空モフラ機器株式会社"]
places = ["架空県雲綴市夢織町", "架空県星架市虹紡町", "架空県月綾市霧紡町"]
openings = [
    "先日の打合せを受け、試作品の納品日と確認手順についてご連絡します。",
    "次回の展示に向けて、見本品の発送予定と受取方法についてご相談します。",
    "追加のご依頼について、検証用機材の準備状況と配送方法をお知らせします。",
]
notes = [
    "到着後は数量と外装をご確認のうえ、同封の確認票に記入してください。",
    "変更がある場合は前日までにお知らせください。担当者が手順を確認します。",
    "資料は会議の前に共有します。確認事項は一覧にまとめてお送りください。",
]
docs = []
for i in range(N_DOCS):
    person, contact = rng.sample(people, 2)
    company = rng.choice(companies)
    place = rng.choice(places)
    phone = f"000-0000-{i + 1:04d}"
    email = f"sample{i:04d}@{rng.choice(['example.com', 'example.jp'])}"
    docs.append(
        f"件名: 試作品の納品確認（架空の連絡文書 {i + 1:04d}）\n"
        f"{company}の{person}です。{rng.choice(openings)}"
        f"納品先は{place}一丁目二番三号、架空ポルネ棟の受付です。"
        f"現地の担当は{contact}、連絡用ハンドルは @sample_{i:04d} です。"
        f"電話 {phone}、メール {email} までご連絡ください。"
        f"{rng.choice(notes)}当日は受付で文書番号をお伝えください。受領後に作業予定を共有します。"
        "本書に記載した人名・組織・連絡先はすべて架空です。"
    )
total_chars = sum(map(len, docs))
print(f"文書数: {len(docs):,} / 平均文字数: {total_chars / len(docs):.1f} / 合計文字数: {total_chars:,}")

In [ ]:
import gc
import time

from privacy_masker import Policy, PrivacyEngine
from privacy_masker.detectors import CharHeadNerDetector, HandleDetector, RegexDetector
from privacy_masker.hub import DEFAULT_MODEL_ID


def make_engine(device, batch_size):
    """Hub の本番 NER + 正規表現 + ハンドル名検出を、連番トークン化で使う。"""
    return PrivacyEngine(
        detectors=[
            CharHeadNerDetector(model=DEFAULT_MODEL_ID, device=device, batch_size=batch_size),
            RegexDetector.default_ja(),
            HandleDetector(),
        ],
        policy=Policy.preset("llm_input"),
    )


def benchmark(docs, device, batch_size):
    """全件でウォームアップ 1 回、本計測 1 回。モデル読み込みは計測外。"""
    engine = make_engine(device, batch_size)
    results = None
    try:
        # ウォームアップの変換結果は保持しない。
        engine.transform_batch(docs, batch_size=batch_size)
        if device == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        results = engine.transform_batch(docs, batch_size=batch_size)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        assert len(results) == len(docs), "変換結果の文書数が一致しない"
        return {
            "device": device,
            "batch_size": batch_size,
            "ms_per_doc": elapsed * 1000 / len(docs),
            "chars_per_sec": sum(map(len, docs)) / elapsed,
            "所要秒": elapsed,
            "peak_memory_MB": torch.cuda.max_memory_allocated() / 1_000_000 if device == "cuda" else None,
        }
    finally:
        # 次の構成にモデルや変換結果を持ち越さない。
        del results, engine
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

In [ ]:
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    # 通常の Python でコードセルだけ確認するときも表を表示する。
    display = print

configs = [("cpu", 16)]
if HAS_CUDA:
    configs += [("cuda", size) for size in (16, 32, 64, 128)]
else:
    print("GPU が無いので CPU だけ測る")

rows = []
for device, batch_size in configs:
    print(f"計測中: device={device}, batch_size={batch_size}", flush=True)
    rows.append(benchmark(docs, device, batch_size))
benchmark_df = pd.DataFrame(rows)
display(benchmark_df.round({"ms_per_doc": 3, "chars_per_sec": 1, "所要秒": 3, "peak_memory_MB": 1}))

In [ ]:
def detection_signatures(device, batch_size):
    """全件の detect() の結果を、型と原文の位置だけにして返す。"""
    engine = make_engine(device, batch_size)
    try:
        return [
            sorted((e.entity_type, e.start, e.end) for e in engine.detect(doc))
            for doc in docs
        ]
    finally:
        del engine
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()


if HAS_CUDA:
    # 速度表とは別に 1 文書ずつ検出して比較する。スコアや返却順は比較しない。
    print(f"CPU と GPU の検出結果を {len(docs):,} 文書で確認中", flush=True)
    cpu_signatures = detection_signatures("cpu", 16)
    gpu_signatures = detection_signatures("cuda", 32)
    mismatches = [
        i for i, (cpu, gpu) in enumerate(zip(cpu_signatures, gpu_signatures)) if cpu != gpu
    ]
    print(f"全件一致: {not mismatches} / 一致文書数: {len(docs) - len(mismatches):,}/{len(docs):,}")
    if mismatches:
        print("不一致の文書番号（1 始まり、先頭 10 件）:", [i + 1 for i in mismatches[:10]])
        i = mismatches[0]
        print("最初の不一致（型, 開始位置, 終了位置）:", {"cpu": cpu_signatures[i], "cuda": gpu_signatures[i]})
else:
    print("GPU が無いので CPU / GPU の検出結果の比較は飛ばす")

## 結果の読み方

- `ms_per_doc` は小さいほど、`chars_per_sec` は大きいほど速い。所要秒は全件の `transform_batch()` にかかった時間。NER が時間のほぼ全部を占める構成だが、この表は正規表現・ハンドル名検出・境界補正・トークン化も含む。
- モデルは既定の float32 で動かす。T4 でも fp16 や autocast は使っていない。GPU の `batch_size` を増やしたときの速度とメモリ使用量を見比べる。
- `peak_memory_MB` は本計測中に PyTorch が割り当てた GPU メモリの最大値（モデルを含む、1 MB = 1,000,000 バイト）。予約済みメモリや GPU 全体の使用量とは異なる。CPU の行は空欄。
- 1 回目のノートブック実行はモデル取得を含むので、セル全体の待ち時間は 2 回目以降を見る。表の数値はモデル取得・読み込みとウォームアップ 1 回を除いた、本計測 1 回だけの値。計測セルを再実行して変動も確認する。
- 検出結果の確認は、既定で全 500 文書の `engine.detect()` の型と位置を CPU / GPU 間で比べる。一致しても検出精度を保証するものではなく、`transform_batch()` の各バッチサイズ間の一致を検証するものでもない。